# MICS DDML analysis

This notebook prepares the household and under-five samples, checks treatment support before estimation, estimates the clustered DoubleML models, and produces the robustness results.

The main steps are intentionally visible. Custom functions are used only where the same nontrivial operation must be repeated: convex Super Learner fitting, PSU-grouped sample splitting, checkpoints, and repeated DoubleML estimation.


## Setup

### Paths and imports


In [1]:
from pathlib import Path
import json
import os
import pickle
from warnings import filterwarnings

import doubleml as dml
import matplotlib.pyplot as plt
import narwhals._interchange
import numpy as np
import pandas as pd
import pyreadstat
import sklearn

from IPython.display import display
from joblib import hash as joblib_hash
from scipy.optimize import minimize
from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin, clone
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import (
    ElasticNetCV,
    LassoCV,
    LinearRegression,
    LogisticRegression,
    LogisticRegressionCV,
    RidgeCV,
)
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from xgboost import XGBClassifier, XGBRegressor

try:
    from doubleml.utils import PSProcessorConfig
except ImportError:
    PSProcessorConfig = None


ROOT = Path("../../").resolve()
DATA = ROOT / "Data" / "3. Final"
OUT = ROOT / "Output"
FIGS = ROOT / "Figures"
TABLES = ROOT / "Writing edit" / "Table"

MODELS = OUT / "models" / "grouped_convex_sl"
OOF_MODELS = MODELS / "oof_propensities"
SUPPORT_MODELS = MODELS / "support_restricted"
LOCO_MODELS = MODELS / "leave_one_country_out"

for folder in [
    OUT,
    FIGS,
    TABLES,
    MODELS,
    OOF_MODELS,
    SUPPORT_MODELS,
    LOCO_MODELS,
]:
    folder.mkdir(parents=True, exist_ok=True)

HH_FILE = DATA / "MASTER_MICS_FINAL.dta"
U5_FILE = DATA / "MASTER_MICS_FINAL_U5.dta"

if not HH_FILE.is_file() or not U5_FILE.is_file():
    raise FileNotFoundError("The HH or U5 source file is missing.")

filterwarnings("ignore")

# Let joblib-based learners use all cores, while avoiding nested BLAS threads.
for variable in [
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
]:
    os.environ[variable] = "1"

os.environ["MPLCONFIGDIR"] = "/tmp/mics_ddml_matplotlib"
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

print(ROOT)


/home/jadrk040507/Dropbox/MICS_DDML


### Settings


In [2]:
SEED = 42
FOLDS = 5
IRM_REPS = 3
APOS_REPS = 3
ROBUSTNESS_REPS = 1
LOCO_REPS = 1
TRIM = 0.01

# Parallelism is kept inside the learners. DoubleML itself runs one model/fold
# at a time to avoid nested process pools and excessive memory use.
DOUBLEML_JOBS = 1

LEVELS = {
    0: "No treatment",
    1: "Boiling",
    2: "Chlorination/tablets",
    3: "Straining/settling",
}

OUTCOME_LABELS = {
    "SomeRiskHome": "Any detectable E. coli at home",
    "VeryHighRiskHome": "Very high E. coli at home (>100 CFU/100 mL)",
    "diarrhea": "Diarrhea among children under five",
}

settings = pd.DataFrame(
    {
        "available_cpus": [os.cpu_count() or 1],
        "folds": [FOLDS],
        "IRM_repetitions": [IRM_REPS],
        "APOS_repetitions": [APOS_REPS],
        "learner_jobs": [-1],
        "DoubleML_jobs": [DOUBLEML_JOBS],
        "propensity_clipping": [TRIM],
    }
)

display(settings)


,available_cpus,folds,IRM_repetitions,APOS_repetitions,learner_jobs,DoubleML_jobs,propensity_clipping
0,16,5,3,3,-1,1,0.01


## Data

### Read the household and under-five files


In [3]:
hh, _ = pyreadstat.read_dta(HH_FILE)
u5, _ = pyreadstat.read_dta(U5_FILE)

hh["_row_id"] = np.arange(len(hh), dtype="int64")
u5["_row_id"] = np.arange(len(u5), dtype="int64")

print(f"HH rows: {len(hh):,}")
print(f"U5 rows: {len(u5):,}")


HH rows: 59,620
U5 rows: 36,121


### Outcomes

`SomeRiskHome` equals one whenever E. coli is detectable. It therefore includes all observations classified as `VeryHighRiskHome`. The latter equals one only above 100 CFU per 100 mL.


In [4]:
for data in [hh, u5]:
    risk = pd.to_numeric(data["RiskHome"], errors="coerce")
    missing_risk = risk.isna()

    data["SomeRiskHome"] = (
        risk.isin([1, 2])
        .mask(missing_risk)
        .astype("Int8")
    )

    data["VeryHighRiskHome"] = (
        risk.eq(2)
        .mask(missing_risk)
        .astype("Int8")
    )

    inconsistent = (
        data["VeryHighRiskHome"].eq(1)
        & ~data["SomeRiskHome"].eq(1)
    )

    assert not inconsistent.any()
    assert data["SomeRiskHome"].isna().equals(
        data["VeryHighRiskHome"].isna()
    )

u5["diarrhea"] = pd.to_numeric(
    u5["diarrhea"],
    errors="coerce",
).astype("Int8")


### Treatment categories

The categories reproduce the grouped coding in the Stata cleaning file:

- `A`: Boiling.
- `B` or `G`: Chlorination/Aquatabs/PUR.
- `C` or `F`: Straining or allowing water to settle.
- `D`, `E`, `H`, and `X`: residual/other methods.

A recognized category takes precedence when a household also reports a residual method. Observations reporting only residual/other methods are removed rather than classified as untreated.


In [5]:
cleaning_rows = []

for sample_name, data in [("HH", hh), ("U5", u5)]:
    method_columns = [
        "WQ15A",
        "WQ15B",
        "WQ15C",
        "WQ15D",
        "WQ15E",
        "WQ15F",
        "WQ15G",
        "WQ15H",
        "WQ15X",
    ]

    missing_columns = sorted(
        set(method_columns + ["WQ15_g"]) - set(data.columns)
    )
    if missing_columns:
        raise KeyError(
            f"{sample_name}: missing treatment variables {missing_columns}"
        )

    used = pd.DataFrame(
        {
            column: (
                data[column]
                .astype("string")
                .str.strip()
                .str.upper()
                .eq(column[-1])
            )
            for column in method_columns
        },
        index=data.index,
    )

    boiling = used["WQ15A"]
    chlorination = used["WQ15B"] | used["WQ15G"]
    straining_settling = used["WQ15C"] | used["WQ15F"]
    residual_other = (
        used["WQ15D"]
        | used["WQ15E"]
        | used["WQ15H"]
        | used["WQ15X"]
    )

    recognized = boiling | chlorination | straining_settling

    response_code = pd.to_numeric(data["WQ15_g"], errors="coerce")
    response_missing = (
        response_code.isna()
        | response_code.isin([99, 998])
    ) & ~used.any(axis=1)

    other_only = residual_other & ~recognized
    no_treatment = ~used.any(axis=1) & ~response_missing

    category = pd.Series(pd.NA, index=data.index, dtype="Int8")
    category.loc[no_treatment] = 0
    category.loc[straining_settling] = 3
    category.loc[chlorination] = 2
    category.loc[boiling] = 1

    binary_treatment = pd.Series(pd.NA, index=data.index, dtype="Int8")
    binary_treatment.loc[no_treatment] = 0
    binary_treatment.loc[recognized] = 1

    data["treatment_count"] = used.sum(axis=1).astype("int16")
    data["multiple_methods"] = data["treatment_count"].gt(1)
    data["other_only"] = other_only
    data["water_treatment"] = binary_treatment
    data["treat_cat"] = category

    cleaning_rows.append(
        {
            "sample": sample_name,
            "rows_before_other_drop": len(data),
            "multiple_method_rows": int(data["multiple_methods"].sum()),
            "multiple_method_percent": (
                100 * data["multiple_methods"].mean()
            ),
            "other_only_rows_dropped": int(other_only.sum()),
            "missing_treatment_rows": int(response_missing.sum()),
        }
    )

    data.drop(index=data.index[other_only], inplace=True)
    data.reset_index(drop=True, inplace=True)

    assert not data["other_only"].any()
    assert set(data["treat_cat"].dropna().unique()).issubset(LEVELS)

cleaning_diagnostics = pd.DataFrame(cleaning_rows)
cleaning_diagnostics.to_csv(
    OUT / "cleaning_diagnostics.csv",
    index=False,
)

display(cleaning_diagnostics)


,sample,rows_before_other_drop,multiple_method_rows,multiple_method_percent,other_only_rows_dropped,missing_treatment_rows
0,HH,59620,1594,2.673599,1357,0
1,U5,36121,695,1.924088,583,0


### PSU and household identifiers

The identifiers always combine country with the survey code. This is simpler and guarantees that repeated numeric codes from different countries are never treated as the same PSU or household.


In [6]:
for sample_name, data in [("HH", hh), ("U5", u5)]:
    if "PSU" in data.columns:
        psu = data["PSU"].copy()
    else:
        psu = pd.Series(pd.NA, index=data.index)

    if "psu" in data.columns:
        psu = psu.where(psu.notna(), data["psu"])

    if "HH1" in data.columns:
        psu = psu.where(psu.notna(), data["HH1"])

    if psu.isna().any():
        raise ValueError(f"{sample_name}: missing PSU identifiers.")

    if "HHID" in data.columns:
        household = data["HHID"].astype("string")
    elif {"HH1", "HH2"}.issubset(data.columns):
        household = (
            data["HH1"].astype("string")
            + "|"
            + data["HH2"].astype("string")
        )
    else:
        raise KeyError(f"{sample_name}: no household identifier found.")

    country = data["country_cat"].astype("string")

    data["_psu_id"] = pd.factorize(
        pd.MultiIndex.from_arrays(
            [country, psu.astype("string")]
        ),
        sort=True,
    )[0].astype("int64")

    data["_hh_id"] = pd.factorize(
        pd.MultiIndex.from_arrays(
            [country, household]
        ),
        sort=True,
    )[0].astype("int64")


### Remaining controls


In [7]:
for data in [hh, u5]:
    for column in [
        "Any_U5",
        "Girls_less_than15",
        "Boys_15or_less",
    ]:
        data[column] = (
            pd.to_numeric(data[column], errors="coerce")
            .fillna(0)
            .astype("int8")
        )

if "age" not in u5.columns or "male" not in u5.columns:
    raise KeyError("The U5 file is missing age or sex.")

u5["child_age"] = pd.to_numeric(u5["age"], errors="coerce")
u5["child_sex_male"] = pd.to_numeric(u5["male"], errors="coerce")


### Construct the four estimation samples

This helper is retained because the same nontrivial complete-case and dummy-variable construction is required for HH/U5 and binary/categorical treatments.


In [8]:
HH_CONTROLS = [
    "windex5",
    "urban",
    "WS1_g",
    "wq27_decile",
    "Any_U5",
    "Girls_less_than15",
    "Boys_15or_less",
    "Toilet",
]

U5_CONTROLS = [
    *HH_CONTROLS,
    "child_age",
    "child_sex_male",
]


def analysis_frame(data, outcomes, treatment, child=False, levels=None):
    raw_controls = U5_CONTROLS if child else HH_CONTROLS

    required = [
        *outcomes,
        treatment,
        "country_cat",
        "_psu_id",
        "_hh_id",
        *raw_controls,
    ]

    keep = data[required].notna().all(axis=1)

    if levels is not None:
        keep &= data[treatment].isin(levels)

    sample = data.loc[keep].copy()

    controls = pd.concat(
        [
            pd.get_dummies(
                sample["windex5"].astype("string"),
                prefix="wealth",
                drop_first=True,
                dtype=float,
            ),
            pd.get_dummies(
                sample["country_cat"].astype("string"),
                prefix="country",
                drop_first=True,
                dtype=float,
            ),
            pd.get_dummies(
                sample["WS1_g"].astype("string"),
                prefix="water_source",
                drop_first=True,
                dtype=float,
            ),
            pd.get_dummies(
                sample["Toilet"].astype("string"),
                prefix="toilet",
                drop_first=True,
                dtype=float,
            ),
            pd.get_dummies(
                sample["wq27_decile"].astype("string"),
                prefix="source_ecoli",
                drop_first=True,
                dtype=float,
            ),
            sample[
                [
                    "urban",
                    "Any_U5",
                    "Girls_less_than15",
                    "Boys_15or_less",
                ]
            ].astype(float),
        ],
        axis=1,
    )

    if child:
        controls = pd.concat(
            [
                controls,
                pd.get_dummies(
                    sample["child_age"].astype("string"),
                    prefix="child_age",
                    drop_first=True,
                    dtype=float,
                ),
                sample[["child_sex_male"]].astype(float),
            ],
            axis=1,
        )

    controls = controls.loc[:, ~controls.columns.duplicated()]

    frame = pd.concat(
        [
            sample[
                [
                    "_row_id",
                    "country_cat",
                    "_psu_id",
                    "_hh_id",
                    *outcomes,
                    treatment,
                ]
            ],
            controls,
        ],
        axis=1,
    ).reset_index(drop=True)

    for outcome in outcomes:
        frame[outcome] = pd.to_numeric(
            frame[outcome],
            errors="raise",
        ).astype(float)

    frame[treatment] = pd.to_numeric(
        frame[treatment],
        errors="raise",
    ).astype(int)

    # The last X column carries PSU only for inner grouped validation.
    # It is removed before the base learners are fitted.
    frame["_psu_model_code"] = pd.factorize(
        frame["_psu_id"],
        sort=True,
    )[0].astype(float)

    x_columns = [
        *controls.columns.tolist(),
        "_psu_model_code",
    ]

    return frame, x_columns


In [9]:
hh_irm, hh_irm_x = analysis_frame(
    hh,
    outcomes=["SomeRiskHome", "VeryHighRiskHome"],
    treatment="water_treatment",
)

hh_apos, hh_apos_x = analysis_frame(
    hh,
    outcomes=["SomeRiskHome", "VeryHighRiskHome"],
    treatment="treat_cat",
    levels=list(LEVELS),
)

u5_irm, u5_irm_x = analysis_frame(
    u5,
    outcomes=["diarrhea"],
    treatment="water_treatment",
    child=True,
)

u5_apos, u5_apos_x = analysis_frame(
    u5,
    outcomes=["diarrhea"],
    treatment="treat_cat",
    child=True,
    levels=list(LEVELS),
)

# With the Stata grouping, the binary and categorical samples should coincide.
assert hh_irm["_row_id"].equals(hh_apos["_row_id"])
assert u5_irm["_row_id"].equals(u5_apos["_row_id"])

analysis_samples = pd.DataFrame(
    [
        {
            "sample": "HH binary",
            "observations": len(hh_irm),
            "PSUs": hh_irm["_psu_id"].nunique(),
            "households": hh_irm["_hh_id"].nunique(),
        },
        {
            "sample": "HH treatment categories",
            "observations": len(hh_apos),
            "PSUs": hh_apos["_psu_id"].nunique(),
            "households": hh_apos["_hh_id"].nunique(),
        },
        {
            "sample": "U5 binary",
            "observations": len(u5_irm),
            "PSUs": u5_irm["_psu_id"].nunique(),
            "households": u5_irm["_hh_id"].nunique(),
        },
        {
            "sample": "U5 treatment categories",
            "observations": len(u5_apos),
            "PSUs": u5_apos["_psu_id"].nunique(),
            "households": u5_apos["_hh_id"].nunique(),
        },
    ]
)

display(analysis_samples)


,sample,observations,PSUs,households
0,HH binary,58263,18585,58263
1,HH treatment categories,58263,18585,58263
2,U5 binary,35538,13408,24733
3,U5 treatment categories,35538,13408,24733


## Model specification

### Learner library


In [10]:
alpha_grid = np.logspace(-3, 3, 7)

REG_LEARNERS = {
    "ols": LinearRegression(),
    "lasso": Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "model",
                LassoCV(
                    cv=3,
                    max_iter=5_000,
                    n_jobs=-1,
                    random_state=SEED,
                ),
            ),
        ]
    ),
    "ridge": Pipeline(
        [
            ("scale", StandardScaler()),
            ("model", RidgeCV(alphas=alpha_grid, cv=3)),
        ]
    ),
    "enet": Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "model",
                ElasticNetCV(
                    cv=3,
                    l1_ratio=[0.5],
                    max_iter=5_000,
                    n_jobs=-1,
                    random_state=SEED,
                ),
            ),
        ]
    ),
    "rf": RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=5,
        random_state=SEED,
        n_jobs=-1,
    ),
    "xgb": XGBRegressor(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        random_state=SEED,
        n_jobs=-1,
        eval_metric="rmse",
    ),
}

CLF_LEARNERS = {
    "logit": Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    penalty=None,
                    solver="lbfgs",
                    max_iter=2_000,
                ),
            ),
        ]
    ),
    "lasso": Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "model",
                LogisticRegressionCV(
                    Cs=alpha_grid,
                    cv=3,
                    penalty="l1",
                    solver="liblinear",
                    scoring="neg_log_loss",
                    max_iter=2_000,
                    n_jobs=-1,
                    random_state=SEED,
                ),
            ),
        ]
    ),
    "ridge": Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "model",
                LogisticRegressionCV(
                    Cs=alpha_grid,
                    cv=3,
                    penalty="l2",
                    solver="lbfgs",
                    scoring="neg_log_loss",
                    max_iter=2_000,
                    n_jobs=-1,
                    random_state=SEED,
                ),
            ),
        ]
    ),
    "enet": Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "model",
                LogisticRegressionCV(
                    Cs=np.logspace(-2, 2, 5),
                    cv=3,
                    penalty="elasticnet",
                    solver="saga",
                    l1_ratios=[0.5],
                    scoring="neg_log_loss",
                    max_iter=3_000,
                    n_jobs=-1,
                    random_state=SEED,
                ),
            ),
        ]
    ),
    "rf": RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=5,
        random_state=SEED,
        n_jobs=-1,
    ),
    "xgb": XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        random_state=SEED,
        n_jobs=-1,
        eval_metric="logloss",
    ),
}


### Convex Super Learner

The meta-learner estimates nonnegative weights that sum to one. Outcome weights minimize OOF squared error; propensity weights minimize OOF Bernoulli log loss.

These classes remain custom because scikit-learn's ordinary stacking estimators do not impose the convex-weight restriction.


In [11]:
def split_X_groups(X):
    X = np.asarray(X)
    return X[:, :-1], X[:, -1].astype("int64")


def inner_splits(y, groups, classification, seed):
    maximum_folds = min(3, np.unique(groups).size)

    for folds in range(maximum_folds, 1, -1):
        if classification:
            splitter = StratifiedGroupKFold(
                n_splits=folds,
                shuffle=True,
                random_state=seed,
            )
            splits = list(
                splitter.split(
                    np.zeros(len(y)),
                    y,
                    groups,
                )
            )
            valid = all(
                np.unique(y[train]).size == 2
                for train, _ in splits
            )
        else:
            splitter = GroupKFold(n_splits=folds)
            splits = list(
                splitter.split(
                    np.zeros(len(y)),
                    y,
                    groups,
                )
            )
            valid = True

        if valid:
            return splits

    raise ValueError(
        "Not enough PSU variation for grouped inner validation."
    )


def convex_weights(predictions, target, loss):
    predictions = np.asarray(predictions, dtype=float)
    target = np.asarray(target, dtype=float)
    n_learners = predictions.shape[1]

    def objective(weights):
        combined = predictions @ weights

        if loss == "mse":
            return np.mean((target - combined) ** 2)

        combined = np.clip(combined, 1e-6, 1 - 1e-6)
        return -np.mean(
            target * np.log(combined)
            + (1 - target) * np.log(1 - combined)
        )

    initial = np.repeat(1 / n_learners, n_learners)

    result = minimize(
        objective,
        initial,
        method="SLSQP",
        bounds=[(0, 1)] * n_learners,
        constraints={
            "type": "eq",
            "fun": lambda weights: weights.sum() - 1,
        },
    )

    if result.success:
        weights = np.clip(result.x, 0, 1)
        return weights / weights.sum()

    # Safe fallback: use the best individual learner.
    losses = []
    for column in range(n_learners):
        candidate = np.zeros(n_learners)
        candidate[column] = 1
        losses.append(objective(candidate))

    weights = np.zeros(n_learners)
    weights[int(np.argmin(losses))] = 1
    return weights


In [12]:
class ConvexSuperLearnerRegressor(RegressorMixin, BaseEstimator):
    def __init__(self, estimators, random_state=42):
        self.estimators = estimators
        self.random_state = random_state

    def fit(self, X, y):
        features, groups = split_X_groups(X)
        target = np.asarray(y, dtype=float)
        splits = inner_splits(
            target,
            groups,
            classification=False,
            seed=self.random_state,
        )

        oof = np.full(
            (len(target), len(self.estimators)),
            np.nan,
        )

        for column, (_, estimator) in enumerate(self.estimators):
            for train, test in splits:
                fitted = clone(estimator)
                fitted.fit(features[train], target[train])
                oof[test, column] = fitted.predict(features[test])

        if np.isnan(oof).any():
            raise RuntimeError("Incomplete outcome OOF predictions.")

        self.weights_ = convex_weights(oof, target, loss="mse")
        self.estimators_ = []

        for name, estimator in self.estimators:
            fitted = clone(estimator)
            fitted.fit(features, target)
            self.estimators_.append((name, fitted))

        self.estimator_names_ = [
            name for name, _ in self.estimators_
        ]
        self.n_features_in_ = np.asarray(X).shape[1]
        return self

    def predict(self, X):
        features, _ = split_X_groups(X)
        predictions = np.column_stack(
            [
                estimator.predict(features)
                for _, estimator in self.estimators_
            ]
        )
        return predictions @ self.weights_


class ConvexSuperLearnerClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, estimators, random_state=42):
        self.estimators = estimators
        self.random_state = random_state

    def fit(self, X, y):
        features, groups = split_X_groups(X)
        target = np.asarray(y, dtype=int)

        if np.unique(target).size != 2:
            raise ValueError(
                "The propensity learner requires two classes."
            )

        splits = inner_splits(
            target,
            groups,
            classification=True,
            seed=self.random_state,
        )

        oof = np.full(
            (len(target), len(self.estimators)),
            np.nan,
        )

        for column, (_, estimator) in enumerate(self.estimators):
            for train, test in splits:
                fitted = clone(estimator)
                fitted.fit(features[train], target[train])
                oof[test, column] = fitted.predict_proba(
                    features[test]
                )[:, 1]

        if np.isnan(oof).any():
            raise RuntimeError(
                "Incomplete propensity OOF predictions."
            )

        self.weights_ = convex_weights(
            oof,
            target,
            loss="log_loss",
        )
        self.estimators_ = []

        for name, estimator in self.estimators:
            fitted = clone(estimator)
            fitted.fit(features, target)
            self.estimators_.append((name, fitted))

        self.estimator_names_ = [
            name for name, _ in self.estimators_
        ]
        self.classes_ = np.array([0, 1])
        self.n_features_in_ = np.asarray(X).shape[1]
        return self

    def predict_proba(self, X):
        features, _ = split_X_groups(X)
        probabilities = np.column_stack(
            [
                estimator.predict_proba(features)[:, 1]
                for _, estimator in self.estimators_
            ]
        )

        positive = np.clip(
            probabilities @ self.weights_,
            1e-8,
            1 - 1e-8,
        )

        return np.column_stack([1 - positive, positive])

    def predict(self, X):
        return (
            self.predict_proba(X)[:, 1] >= 0.5
        ).astype(int)


ML_G = ConvexSuperLearnerRegressor(
    estimators=list(REG_LEARNERS.items()),
    random_state=SEED,
)

ML_M = ConvexSuperLearnerClassifier(
    estimators=list(CLF_LEARNERS.items()),
    random_state=SEED,
)


## Pre-estimation diagnostics

### Treatment frequencies and survey structure


In [13]:
frequency_rows = []
structure_rows = []

for sample_name, frame in [("HH", hh_apos), ("U5", u5_apos)]:
    frequencies = (
        frame.groupby("treat_cat")
        .agg(
            observations=("_row_id", "size"),
            PSUs=("_psu_id", "nunique"),
            households=("_hh_id", "nunique"),
        )
        .reset_index()
        .rename(columns={"treat_cat": "treatment_code"})
    )

    frequencies["sample"] = sample_name
    frequencies["treatment_category"] = (
        frequencies["treatment_code"].map(LEVELS)
    )
    frequencies["share"] = (
        frequencies["observations"]
        / frequencies["observations"].sum()
    )
    frequency_rows.append(frequencies)

    psu_sizes = frame.groupby("_psu_id").size()
    household_sizes = frame.groupby("_hh_id").size()

    structure_rows.append(
        {
            "sample": sample_name,
            "observations": len(frame),
            "PSUs": frame["_psu_id"].nunique(),
            "households": frame["_hh_id"].nunique(),
            "mean_observations_per_PSU": psu_sizes.mean(),
            "median_observations_per_PSU": psu_sizes.median(),
            "p90_observations_per_PSU": psu_sizes.quantile(0.90),
            "maximum_observations_per_PSU": psu_sizes.max(),
            "mean_observations_per_household": household_sizes.mean(),
            "median_observations_per_household": household_sizes.median(),
        }
    )

treatment_frequencies = pd.concat(
    frequency_rows,
    ignore_index=True,
)

survey_structure = pd.DataFrame(structure_rows)

treatment_frequencies.to_csv(
    OUT / "treatment_category_frequencies.csv",
    index=False,
)
survey_structure.to_csv(
    OUT / "psu_structure_summary.csv",
    index=False,
)

display(treatment_frequencies)
display(survey_structure)


,treatment_code,observations,PSUs,households,sample,treatment_category,share
0,0,46951,16973,46951,HH,No treatment,0.805846
1,1,7238,3070,7238,HH,Boiling,0.124230
2,2,1442,1149,1442,HH,Chlorination/tablets,0.024750
3,3,2632,1227,2632,HH,Straining/settling,0.045174
4,0,29097,11567,20247,U5,No treatment,0.818757
5,1,3216,1750,2468,U5,Boiling,0.090495
6,2,831,515,587,U5,Chlorination/tablets,0.023383
7,3,2394,791,1431,U5,Straining/settling,0.067365


,sample,observations,PSUs,households,mean_observations_per_PSU,median_observations_per_PSU,p90_observations_per_PSU,maximum_observations_per_PSU,mean_observations_per_household,median_observations_per_household
0,HH,58263,18585,58263,3.134948,3.0,5.0,8,1.000000,1.0
1,U5,35538,13408,24733,2.650507,2.0,5.0,25,1.436866,1.0


### Observed support by country and PSU

The basic condition requires both `No treatment` and the relevant category within a country. The stricter diagnostic requires each category to occur in at least two distinct PSU. It does not require each individual PSU to contain both categories.


In [14]:
def support_by_country(frame, sample_name):
    counts = (
        frame.groupby(["country_cat", "treat_cat"])
        .agg(
            observations=("_row_id", "size"),
            PSUs=("_psu_id", "nunique"),
        )
        .reset_index()
    )

    lookup = counts.set_index(
        ["country_cat", "treat_cat"]
    )
    countries = sorted(frame["country_cat"].unique())

    detail = []
    summary = []

    for level in [1, 2, 3]:
        comparison = []

        for country in countries:
            key_control = (country, 0)
            key_treated = (country, level)

            n_control = (
                int(lookup.loc[key_control, "observations"])
                if key_control in lookup.index
                else 0
            )
            psu_control = (
                int(lookup.loc[key_control, "PSUs"])
                if key_control in lookup.index
                else 0
            )
            n_treated = (
                int(lookup.loc[key_treated, "observations"])
                if key_treated in lookup.index
                else 0
            )
            psu_treated = (
                int(lookup.loc[key_treated, "PSUs"])
                if key_treated in lookup.index
                else 0
            )

            row = {
                "sample": sample_name,
                "country": country,
                "treatment_code": level,
                "treatment_category": LEVELS[level],
                "N_no_treatment": n_control,
                "PSU_no_treatment": psu_control,
                "N_category": n_treated,
                "PSU_category": psu_treated,
                "both_categories_present": (
                    n_control > 0 and n_treated > 0
                ),
                "at_least_2_PSU_each": (
                    psu_control >= 2 and psu_treated >= 2
                ),
            }

            detail.append(row)
            comparison.append(row)

        comparison = pd.DataFrame(comparison)
        treated = frame.loc[frame["treat_cat"].eq(level)]

        summary.append(
            {
                "sample": sample_name,
                "treatment_category": LEVELS[level],
                "observations": len(treated),
                "PSUs": treated["_psu_id"].nunique(),
                "countries_with_category": (
                    treated["country_cat"].nunique()
                ),
                "countries_with_category_and_no_treatment": (
                    comparison["both_categories_present"].sum()
                ),
                "countries_with_at_least_2_PSU_each": (
                    comparison["at_least_2_PSU_each"].sum()
                ),
            }
        )

    return pd.DataFrame(detail), pd.DataFrame(summary)


hh_support_detail, hh_support_summary = support_by_country(
    hh_apos,
    "HH",
)
u5_support_detail, u5_support_summary = support_by_country(
    u5_apos,
    "U5",
)

support_detail = pd.concat(
    [hh_support_detail, u5_support_detail],
    ignore_index=True,
)
support_summary = pd.concat(
    [hh_support_summary, u5_support_summary],
    ignore_index=True,
)

support_detail.to_csv(
    OUT / "positivity_support_by_country.csv",
    index=False,
)
support_summary.to_csv(
    OUT / "positivity_support_summary.csv",
    index=False,
)

display(support_summary)


,sample,treatment_category,observations,PSUs,countries_with_category,countries_with_category_and_no_treatment,countries_with_at_least_2_PSU_each
0,HH,Boiling,7238,3070,25,25,24
1,HH,Chlorination/tablets,1442,1149,24,24,22
2,HH,Straining/settling,2632,1227,25,25,25
3,U5,Boiling,3216,1750,24,24,22
4,U5,Chlorination/tablets,831,515,21,21,19
5,U5,Straining/settling,2394,791,25,25,24


### PSU-grouped cross-fitting folds

This is one of the places where a helper materially improves correctness: every PSU and household must remain wholly in training or test, every observation must appear once in test per repetition, and every training fold must retain all treatment categories.


In [15]:
def grouped_folds(frame, treatment, repetitions, seed):
    y = frame[treatment].to_numpy(dtype=int)
    groups = frame["_psu_id"].to_numpy(dtype=int)
    households = frame["_hh_id"].to_numpy(dtype=int)
    required_levels = np.sort(np.unique(y))

    all_smpls = []
    all_cluster_smpls = []
    audit = []

    for repetition in range(repetitions):
        selected = None

        for attempt in range(100):
            splitter = StratifiedGroupKFold(
                n_splits=FOLDS,
                shuffle=True,
                random_state=seed + 1_000 * repetition + attempt,
            )

            candidate = list(
                splitter.split(
                    np.zeros(len(frame)),
                    y,
                    groups,
                )
            )

            valid = all(
                np.array_equal(
                    np.sort(np.unique(y[train])),
                    required_levels,
                )
                for train, _ in candidate
            )

            if valid:
                selected = candidate
                break

        if selected is None:
            raise ValueError(
                "No grouped split retained every treatment "
                "category in every training fold."
            )

        test_frequency = np.zeros(len(frame), dtype=int)
        repetition_smpls = []
        repetition_cluster_smpls = []

        for fold, (train, test) in enumerate(selected):
            train = np.asarray(train, dtype=int)
            test = np.asarray(test, dtype=int)

            train_psu = np.unique(groups[train])
            test_psu = np.unique(groups[test])
            train_hh = np.unique(households[train])
            test_hh = np.unique(households[test])

            assert not np.intersect1d(train_psu, test_psu).size
            assert not np.intersect1d(train_hh, test_hh).size

            test_frequency[test] += 1
            repetition_smpls.append((train, test))
            repetition_cluster_smpls.append(
                ([train_psu], [test_psu])
            )

            row = {
                "repetition": repetition,
                "fold": fold,
                "train_observations": len(train),
                "test_observations": len(test),
                "train_PSUs": len(train_psu),
                "test_PSUs": len(test_psu),
                "train_households": len(train_hh),
                "test_households": len(test_hh),
                "test_countries": (
                    frame.iloc[test]["country_cat"].nunique()
                ),
            }

            for level in required_levels:
                row[f"test_share_{level}"] = np.mean(
                    y[test] == level
                )

            audit.append(row)

        assert np.all(test_frequency == 1)

        all_smpls.append(repetition_smpls)
        all_cluster_smpls.append(
            repetition_cluster_smpls
        )

    return (
        all_smpls,
        all_cluster_smpls,
        pd.DataFrame(audit),
    )


In [16]:
hh_irm_smpls, hh_irm_cluster_smpls, hh_irm_audit = grouped_folds(
    hh_irm,
    "water_treatment",
    IRM_REPS,
    SEED,
)

hh_apos_smpls, hh_apos_cluster_smpls, hh_apos_audit = grouped_folds(
    hh_apos,
    "treat_cat",
    APOS_REPS,
    SEED,
)

u5_irm_smpls, u5_irm_cluster_smpls, u5_irm_audit = grouped_folds(
    u5_irm,
    "water_treatment",
    IRM_REPS,
    SEED,
)

u5_apos_smpls, u5_apos_cluster_smpls, u5_apos_audit = grouped_folds(
    u5_apos,
    "treat_cat",
    APOS_REPS,
    SEED,
)

fold_audit = pd.concat(
    [
        hh_irm_audit.assign(sample="HH", model="IRM"),
        hh_apos_audit.assign(sample="HH", model="APOS"),
        u5_irm_audit.assign(sample="U5", model="IRM"),
        u5_apos_audit.assign(sample="U5", model="APOS"),
    ],
    ignore_index=True,
)

fold_audit.to_csv(
    OUT / "fold_balance_grouped_convex_sl.csv",
    index=False,
)

display(fold_audit)


,repetition,fold,train_observations,test_observations,train_PSUs,test_PSUs,train_households,test_households,test_countries,test_share_0,test_share_1,sample,model,test_share_2,test_share_3
0,0,0,46610,11653,14875,3710,46610,11653,25,0.805887,0.194113,HH,IRM,NaN,NaN
1,0,1,46612,11651,14880,3705,46612,11651,25,0.805854,0.194146,HH,IRM,NaN,NaN
2,0,2,46609,11654,14876,3709,46609,11654,25,0.805818,0.194182,HH,IRM,NaN,NaN
3,0,3,46610,11653,14833,3752,46610,11653,25,0.805801,0.194199,HH,IRM,NaN,NaN
4,0,4,46611,11652,14876,3709,46611,11652,25,0.805870,0.194130,HH,IRM,NaN,NaN
5,1,0,46610,11653,14868,3717,46610,11653,25,0.805887,0.194113,HH,IRM,NaN,NaN
6,1,1,46611,11652,14874,3711,46611,11652,25,0.805870,0.194130,HH,IRM,NaN,NaN
7,1,2,46609,11654,14890,3695,46609,11654,25,0.805818,0.194182,HH,IRM,NaN,NaN
8,1,3,46611,11652,14850,3735,46611,11652,25,0.805870,0.194130,HH,IRM,NaN,NaN
9,1,4,46611,11652,14858,3727,46611,11652,25,0.805784,0.194216,HH,IRM,NaN,NaN


### Checkpoints

A single compact checkpoint helper replaces the previous chain of wrappers. A cached object is used only when its exact specification signature matches.


In [17]:
def load_or_fit(path, signature, fit):
    if path.is_file():
        try:
            with path.open("rb") as file:
                cached = pickle.load(file)

            if cached.get("signature") == signature:
                tqdm.write(f"Loaded {path.name}")
                return cached["value"]
        except Exception:
            pass

    value = fit()
    temporary = path.with_suffix(path.suffix + ".tmp")

    with temporary.open("wb") as file:
        pickle.dump(
            {
                "signature": signature,
                "value": value,
            },
            file,
            protocol=pickle.HIGHEST_PROTOCOL,
        )
        file.flush()
        os.fsync(file.fileno())

    os.replace(temporary, path)
    tqdm.write(f"Saved {path.name}")
    return value


def specification_signature(
    kind,
    frame,
    outcomes,
    treatment,
    x_columns,
    splits,
    ml_g=ML_G,
    ml_m=ML_M,
):
    relevant_columns = [
        "_row_id",
        "country_cat",
        "_psu_id",
        "_hh_id",
        *outcomes,
        treatment,
        *x_columns,
    ]

    return joblib_hash(
        {
            "kind": kind,
            "data": joblib_hash(frame[relevant_columns]),
            "outcomes": outcomes,
            "treatment": treatment,
            "x_columns": x_columns,
            "splits": splits,
            "ml_g": ml_g,
            "ml_m": ml_m,
            "trim": TRIM,
            "seed": SEED,
            "doubleml": dml.__version__,
            "sklearn": sklearn.__version__,
        }
    )


### Raw out-of-fold propensity scores

This estimates only the treatment nuisance. Raw predictions are used for positivity diagnostics; clipped predictions are stored separately for comparison with the score threshold.


In [18]:
def oof_propensities(
    frame,
    treatment,
    x_columns,
    splits,
    levels,
    sample_name,
    model_name,
):
    X = frame[x_columns].to_numpy(dtype=float)
    observed = frame[treatment].to_numpy(dtype=int)

    prediction_rows = []
    weight_rows = []

    for repetition, repetition_splits in enumerate(splits):
        for level in levels:
            predictions = np.full(len(frame), np.nan)

            for fold, (train, test) in enumerate(repetition_splits):
                learner = clone(ML_M)
                target = (observed[train] == level).astype(int)

                learner.fit(X[train], target)
                predictions[test] = learner.predict_proba(
                    X[test]
                )[:, 1]

                for name, weight in zip(
                    learner.estimator_names_,
                    learner.weights_,
                ):
                    weight_rows.append(
                        {
                            "sample": sample_name,
                            "model": model_name,
                            "treatment_code": level,
                            "treatment_category": LEVELS.get(
                                level,
                                str(level),
                            ),
                            "repetition": repetition,
                            "fold": fold,
                            "learner": name,
                            "weight": weight,
                        }
                    )

            if np.isnan(predictions).any():
                raise RuntimeError(
                    "Incomplete outer propensity predictions."
                )

            clipped = np.clip(
                predictions,
                TRIM,
                1 - TRIM,
            )

            block = pd.DataFrame(
                {
                    "sample": sample_name,
                    "model": model_name,
                    "row_id": frame["_row_id"],
                    "country": frame["country_cat"],
                    "PSU": frame["_psu_id"],
                    "observed_treatment": observed,
                    "treatment_code": level,
                    "treatment_category": LEVELS.get(
                        level,
                        str(level),
                    ),
                    "repetition": repetition,
                    "propensity_raw": predictions,
                    "propensity_clipped": clipped,
                }
            )

            prediction_rows.append(block)

    return (
        pd.concat(prediction_rows, ignore_index=True),
        pd.DataFrame(weight_rows),
    )


In [ ]:
oof_specs = [
    {
        "name": "hh_irm",
        "frame": hh_irm,
        "treatment": "water_treatment",
        "x": hh_irm_x,
        "splits": hh_irm_smpls,
        "levels": [1],
        "sample": "HH",
        "model": "Any recognized treatment",
    },
    {
        "name": "hh_apos",
        "frame": hh_apos,
        "treatment": "treat_cat",
        "x": hh_apos_x,
        "splits": hh_apos_smpls,
        "levels": list(LEVELS),
        "sample": "HH",
        "model": "Treatment categories",
    },
    {
        "name": "u5_irm",
        "frame": u5_irm,
        "treatment": "water_treatment",
        "x": u5_irm_x,
        "splits": u5_irm_smpls,
        "levels": [1],
        "sample": "U5",
        "model": "Any recognized treatment",
    },
    {
        "name": "u5_apos",
        "frame": u5_apos,
        "treatment": "treat_cat",
        "x": u5_apos_x,
        "splits": u5_apos_smpls,
        "levels": list(LEVELS),
        "sample": "U5",
        "model": "Treatment categories",
    },
]

oof_tables = []
oof_weight_tables = []

for spec in oof_specs:
    signature = joblib_hash(
        {
            "data": joblib_hash(
                spec["frame"][
                    [
                        "_row_id",
                        "country_cat",
                        "_psu_id",
                        spec["treatment"],
                        *spec["x"],
                    ]
                ]
            ),
            "treatment": spec["treatment"],
            "x": spec["x"],
            "splits": spec["splits"],
            "levels": spec["levels"],
            "learner": ML_M,
            "trim": TRIM,
        }
    )

    predictions, weights = load_or_fit(
        OOF_MODELS / f'{spec["name"]}.pkl',
        signature,
        lambda spec=spec: oof_propensities(
            spec["frame"],
            spec["treatment"],
            spec["x"],
            spec["splits"],
            spec["levels"],
            spec["sample"],
            spec["model"],
        ),
    )

    oof_tables.append(predictions)
    oof_weight_tables.append(weights)

propensity_oof = pd.concat(
    oof_tables,
    ignore_index=True,
)
propensity_weights = pd.concat(
    oof_weight_tables,
    ignore_index=True,
)

propensity_oof.to_csv(
    OUT / "propensity_oof_raw_and_clipped.csv",
    index=False,
)
propensity_weights.to_csv(
    OUT / "super_learner_propensity_weights.csv",
    index=False,
)


Saved hh_irm.pkl


### Propensity summary and plots


In [ ]:
propensity_rows = []

group_columns = [
    "sample",
    "model",
    "treatment_code",
    "treatment_category",
]

for keys, group in propensity_oof.groupby(group_columns):
    values = group["propensity_raw"]
    quantiles = values.quantile(
        [0.01, 0.05, 0.50, 0.95, 0.99]
    )

    propensity_rows.append(
        {
            "sample": keys[0],
            "model": keys[1],
            "treatment_code": keys[2],
            "treatment_category": keys[3],
            "p01": quantiles.loc[0.01],
            "p05": quantiles.loc[0.05],
            "median": quantiles.loc[0.50],
            "p95": quantiles.loc[0.95],
            "p99": quantiles.loc[0.99],
            "below_0.01_percent": (
                100 * values.lt(0.01).mean()
            ),
            "above_0.99_percent": (
                100 * values.gt(0.99).mean()
            ),
            "clipped_percent": (
                100
                * group["propensity_raw"]
                .ne(group["propensity_clipped"])
                .mean()
            ),
        }
    )

propensity_summary = pd.DataFrame(propensity_rows)
propensity_summary.to_csv(
    OUT / "propensity_oof_summary.csv",
    index=False,
)

display(propensity_summary)


In [ ]:
for sample_name in ["HH", "U5"]:
    for level in [1, 2, 3]:
        data = propensity_oof.loc[
            propensity_oof["sample"].eq(sample_name)
            & propensity_oof["model"].eq(
                "Treatment categories"
            )
            & propensity_oof["treatment_code"].eq(level)
        ]

        values = np.sort(
            data["propensity_raw"].to_numpy()
        )
        cumulative = (
            np.arange(1, len(values) + 1) / len(values)
        )

        plt.figure(figsize=(7, 4.5))
        plt.plot(values, cumulative)
        plt.axvline(TRIM, linestyle="--")
        plt.axvline(1 - TRIM, linestyle="--")
        plt.xlabel("Raw out-of-fold propensity")
        plt.ylabel("Empirical cumulative probability")
        plt.title(f"{sample_name}: {LEVELS[level]}")
        plt.tight_layout()
        plt.savefig(
            FIGS
            / (
                f"propensity_ecdf_"
                f"{sample_name.lower()}_{level}.png"
            ),
            dpi=200,
        )
        plt.show()


### Diagnostic checkpoint

At this point, the notebook has checked outcome coding, treatment coding, survey structure, observed support, fold integrity, and raw OOF propensities. No causal effect has yet been estimated.


In [ ]:
diagnostic_checkpoint = pd.Series(
    {
        "HH_observations": len(hh_irm),
        "HH_PSUs": hh_irm["_psu_id"].nunique(),
        "U5_observations": len(u5_irm),
        "U5_PSUs": u5_irm["_psu_id"].nunique(),
        "raw_propensity_rows": len(propensity_oof),
        "maximum_share_below_0.01": (
            propensity_summary["below_0.01_percent"].max()
        ),
    },
    name="value",
)

display(diagnostic_checkpoint)


## Estimation

Only two model-fitting helpers are retained because the same specification is repeatedly used for the main estimates, support restrictions, and LOCO.


In [ ]:
if PSProcessorConfig is not None:
    PS_KWARGS = {
        "ps_processor_config": PSProcessorConfig(
            clipping_threshold=TRIM,
        )
    }
else:
    PS_KWARGS = {
        "trimming_rule": "truncate",
        "trimming_threshold": TRIM,
    }


def fit_irm(
    frame,
    outcome,
    treatment,
    x_columns,
    splits,
    cluster_splits,
    ml_g=ML_G,
    ml_m=ML_M,
):
    data = dml.DoubleMLData(
        frame,
        y_col=outcome,
        d_cols=treatment,
        x_cols=x_columns,
        cluster_cols="_psu_id",
    )

    model = dml.DoubleMLIRM(
        data,
        ml_g=clone(ml_g),
        ml_m=clone(ml_m),
        score="ATE",
        draw_sample_splitting=False,
        **PS_KWARGS,
    )

    model.set_sample_splitting(
        splits,
        cluster_splits,
    )

    model.fit(
        n_jobs_cv=DOUBLEML_JOBS,
        store_predictions=True,
        store_models=False,
    )

    return model


def fit_apos(
    frame,
    outcome,
    treatment,
    x_columns,
    splits,
    cluster_splits,
    ml_g=ML_G,
    ml_m=ML_M,
):
    data = dml.DoubleMLData(
        frame,
        y_col=outcome,
        d_cols=treatment,
        x_cols=x_columns,
        cluster_cols="_psu_id",
    )

    model = dml.DoubleMLAPOS(
        data,
        ml_g=clone(ml_g),
        ml_m=clone(ml_m),
        treatment_levels=list(LEVELS),
        draw_sample_splitting=False,
        **PS_KWARGS,
    )

    model.set_sample_splitting(
        splits,
        cluster_splits,
    )

    model.fit(
        n_jobs_models=DOUBLEML_JOBS,
        n_jobs_cv=DOUBLEML_JOBS,
        store_predictions=True,
        store_models=False,
    )

    return {
        "model": model,
        "contrast": model.causal_contrast(
            reference_levels=0
        ),
    }


### Main IRM estimates


In [ ]:
irm_specs = [
    {
        "outcome": "SomeRiskHome",
        "frame": hh_irm,
        "x": hh_irm_x,
        "splits": hh_irm_smpls,
        "cluster_splits": hh_irm_cluster_smpls,
        "filename": "irm_hh_some_risk.pkl",
    },
    {
        "outcome": "VeryHighRiskHome",
        "frame": hh_irm,
        "x": hh_irm_x,
        "splits": hh_irm_smpls,
        "cluster_splits": hh_irm_cluster_smpls,
        "filename": "irm_hh_very_high_risk.pkl",
    },
    {
        "outcome": "diarrhea",
        "frame": u5_irm,
        "x": u5_irm_x,
        "splits": u5_irm_smpls,
        "cluster_splits": u5_irm_cluster_smpls,
        "filename": "irm_u5_diarrhea.pkl",
    },
]

irm_models = {}

for spec in irm_specs:
    signature = specification_signature(
        "IRM",
        spec["frame"],
        [spec["outcome"]],
        "water_treatment",
        spec["x"],
        spec["splits"],
    )

    irm_models[spec["outcome"]] = load_or_fit(
        MODELS / spec["filename"],
        signature,
        lambda spec=spec: fit_irm(
            spec["frame"],
            spec["outcome"],
            "water_treatment",
            spec["x"],
            spec["splits"],
            spec["cluster_splits"],
        ),
    )


### Main treatment-category estimates


In [ ]:
apos_specs = [
    {
        "outcome": "SomeRiskHome",
        "frame": hh_apos,
        "x": hh_apos_x,
        "splits": hh_apos_smpls,
        "cluster_splits": hh_apos_cluster_smpls,
        "filename": "apos_hh_some_risk.pkl",
    },
    {
        "outcome": "VeryHighRiskHome",
        "frame": hh_apos,
        "x": hh_apos_x,
        "splits": hh_apos_smpls,
        "cluster_splits": hh_apos_cluster_smpls,
        "filename": "apos_hh_very_high_risk.pkl",
    },
    {
        "outcome": "diarrhea",
        "frame": u5_apos,
        "x": u5_apos_x,
        "splits": u5_apos_smpls,
        "cluster_splits": u5_apos_cluster_smpls,
        "filename": "apos_u5_diarrhea.pkl",
    },
]

apos_models = {}

for spec in apos_specs:
    signature = specification_signature(
        "APOS",
        spec["frame"],
        [spec["outcome"]],
        "treat_cat",
        spec["x"],
        spec["splits"],
    )

    apos_models[spec["outcome"]] = load_or_fit(
        MODELS / spec["filename"],
        signature,
        lambda spec=spec: fit_apos(
            spec["frame"],
            spec["outcome"],
            "treat_cat",
            spec["x"],
            spec["splits"],
            spec["cluster_splits"],
        ),
    )


## Results

### Main estimates


In [ ]:
main_rows = []

for outcome, model in irm_models.items():
    row = model.summary.iloc[0]

    main_rows.append(
        {
            "model": "Any recognized treatment",
            "outcome": outcome,
            "outcome_label": OUTCOME_LABELS[outcome],
            "comparison": (
                "Any recognized treatment vs no treatment"
            ),
            "coefficient": row.get("coef", np.nan),
            "standard_error": row.get("std err", np.nan),
            "p_value": row.get("P>|t|", np.nan),
            "ci_low": row.get("2.5 %", np.nan),
            "ci_high": row.get("97.5 %", np.nan),
        }
    )

for outcome, bundle in apos_models.items():
    for index, row in bundle["contrast"].summary.iterrows():
        level = int(float(str(index).split(" vs ")[0]))

        main_rows.append(
            {
                "model": "Treatment categories",
                "outcome": outcome,
                "outcome_label": OUTCOME_LABELS[outcome],
                "comparison": (
                    f"{LEVELS[level]} vs no treatment"
                ),
                "treatment_code": level,
                "treatment_category": LEVELS[level],
                "coefficient": row.get("coef", np.nan),
                "standard_error": row.get(
                    "std err",
                    np.nan,
                ),
                "p_value": row.get("P>|t|", np.nan),
                "ci_low": row.get("2.5 %", np.nan),
                "ci_high": row.get("97.5 %", np.nan),
            }
        )

main_results = pd.DataFrame(main_rows)
main_results.to_csv(
    OUT / "results_main_grouped_convex_sl.csv",
    index=False,
)

display(main_results)


### Super Learner propensity weights

These are convex-combination weights for the nuisance learner, not inverse-probability weights.


In [ ]:
propensity_weight_summary = (
    propensity_weights.groupby(
        [
            "sample",
            "model",
            "treatment_category",
            "learner",
        ]
    )["weight"]
    .agg(["mean", "median", "std", "min", "max"])
    .reset_index()
)

propensity_weight_summary.to_csv(
    OUT / "super_learner_propensity_weight_summary.csv",
    index=False,
)

display(propensity_weight_summary)


## Robustness

### Support-restricted pairwise comparisons

The first rule retains countries containing both categories. The stricter rule requires at least two PSU in each category. These restrictions change the target population, so they are robustness analyses rather than the main estimates.


In [ ]:
support_results = []

support_specs = [
    ("HH", hh, "SomeRiskHome", hh_support_detail, False),
    ("HH", hh, "VeryHighRiskHome", hh_support_detail, False),
    ("U5", u5, "diarrhea", u5_support_detail, True),
]

for sample_name, data, outcome, detail, child in support_specs:
    for level in [1, 2, 3]:
        level_support = detail.loc[
            detail["treatment_code"].eq(level)
        ]

        for minimum_psu in [1, 2]:
            if minimum_psu == 1:
                eligible = set(
                    level_support.loc[
                        level_support[
                            "both_categories_present"
                        ],
                        "country",
                    ]
                )
                rule_label = "Both categories present"
            else:
                eligible = set(
                    level_support.loc[
                        level_support[
                            "at_least_2_PSU_each"
                        ],
                        "country",
                    ]
                )
                rule_label = (
                    "At least two PSU in each category"
                )

            pair = data.loc[
                data["country_cat"].isin(eligible)
                & data["treat_cat"].isin([0, level])
            ].copy()

            pair["pair_treatment"] = (
                pair["treat_cat"].eq(level).astype("Int8")
            )

            try:
                frame, x_columns = analysis_frame(
                    pair,
                    outcomes=[outcome],
                    treatment="pair_treatment",
                    child=child,
                )

                splits, cluster_splits, _ = grouped_folds(
                    frame,
                    "pair_treatment",
                    ROBUSTNESS_REPS,
                    SEED + 10_000 + level + minimum_psu,
                )

                filename = (
                    f"{sample_name.lower()}_{outcome}_"
                    f"level_{level}_psu_{minimum_psu}.pkl"
                )

                signature = specification_signature(
                    "SUPPORT_IRM",
                    frame,
                    [outcome],
                    "pair_treatment",
                    x_columns,
                    splits,
                )

                model = load_or_fit(
                    SUPPORT_MODELS / filename,
                    signature,
                    lambda: fit_irm(
                        frame,
                        outcome,
                        "pair_treatment",
                        x_columns,
                        splits,
                        cluster_splits,
                    ),
                )

                row = model.summary.iloc[0]

                support_results.append(
                    {
                        "sample": sample_name,
                        "outcome": outcome,
                        "treatment_code": level,
                        "treatment_category": LEVELS[level],
                        "comparison": (
                            f"{LEVELS[level]} vs no treatment"
                        ),
                        "support_rule": rule_label,
                        "countries": len(eligible),
                        "observations": len(frame),
                        "PSUs": frame["_psu_id"].nunique(),
                        "coefficient": row.get("coef", np.nan),
                        "standard_error": row.get(
                            "std err",
                            np.nan,
                        ),
                        "p_value": row.get(
                            "P>|t|",
                            np.nan,
                        ),
                    }
                )

            except ValueError as error:
                support_results.append(
                    {
                        "sample": sample_name,
                        "outcome": outcome,
                        "treatment_code": level,
                        "treatment_category": LEVELS[level],
                        "support_rule": rule_label,
                        "status": str(error),
                    }
                )

support_results = pd.DataFrame(support_results)
support_results.to_csv(
    OUT / "results_support_restricted.csv",
    index=False,
)

display(support_results)


### Leave-one-country-out

Country indicators, grouped folds, and nuisance functions are rebuilt after each exclusion. Only the convex Super Learner specification is used.


In [ ]:
loco_rows = []

loco_specs = [
    ("HH", hh, ["SomeRiskHome", "VeryHighRiskHome"], False),
    ("U5", u5, ["diarrhea"], True),
]

for sample_name, data, outcomes, child in loco_specs:
    countries = sorted(
        data["country_cat"].dropna().unique()
    )

    for country in tqdm(
        countries,
        desc=f"LOCO {sample_name}",
        leave=False,
    ):
        reduced = data.loc[
            ~data["country_cat"].eq(country)
        ].copy()

        # Binary treatment models
        binary_frame, binary_x = analysis_frame(
            reduced,
            outcomes=outcomes,
            treatment="water_treatment",
            child=child,
        )

        binary_splits, binary_cluster_splits, _ = (
            grouped_folds(
                binary_frame,
                "water_treatment",
                LOCO_REPS,
                SEED
                + 20_000
                + int(joblib_hash(country)[:8], 16),
            )
        )

        for outcome in outcomes:
            filename = (
                f"irm_{sample_name.lower()}_{outcome}_"
                f"without_{country}.pkl"
            )

            signature = specification_signature(
                "LOCO_IRM",
                binary_frame,
                [outcome],
                "water_treatment",
                binary_x,
                binary_splits,
            )

            model = load_or_fit(
                LOCO_MODELS / filename,
                signature,
                lambda outcome=outcome: fit_irm(
                    binary_frame,
                    outcome,
                    "water_treatment",
                    binary_x,
                    binary_splits,
                    binary_cluster_splits,
                ),
            )

            row = model.summary.iloc[0]

            loco_rows.append(
                {
                    "sample": sample_name,
                    "outcome": outcome,
                    "comparison": (
                        "Any recognized treatment "
                        "vs no treatment"
                    ),
                    "excluded_country": country,
                    "observations": len(binary_frame),
                    "PSUs": (
                        binary_frame["_psu_id"].nunique()
                    ),
                    "households": (
                        binary_frame["_hh_id"].nunique()
                    ),
                    "coefficient": row.get("coef", np.nan),
                    "standard_error": row.get(
                        "std err",
                        np.nan,
                    ),
                    "p_value": row.get(
                        "P>|t|",
                        np.nan,
                    ),
                }
            )

        # Treatment-category models
        category_frame, category_x = analysis_frame(
            reduced,
            outcomes=outcomes,
            treatment="treat_cat",
            child=child,
            levels=list(LEVELS),
        )

        if set(category_frame["treat_cat"].unique()) != set(LEVELS):
            for outcome in outcomes:
                loco_rows.append(
                    {
                        "sample": sample_name,
                        "outcome": outcome,
                        "excluded_country": country,
                        "status": (
                            "At least one treatment "
                            "category disappears."
                        ),
                    }
                )
            continue

        category_splits, category_cluster_splits, _ = (
            grouped_folds(
                category_frame,
                "treat_cat",
                LOCO_REPS,
                SEED
                + 30_000
                + int(joblib_hash(country)[:8], 16),
            )
        )

        for outcome in outcomes:
            filename = (
                f"apos_{sample_name.lower()}_{outcome}_"
                f"without_{country}.pkl"
            )

            signature = specification_signature(
                "LOCO_APOS",
                category_frame,
                [outcome],
                "treat_cat",
                category_x,
                category_splits,
            )

            bundle = load_or_fit(
                LOCO_MODELS / filename,
                signature,
                lambda outcome=outcome: fit_apos(
                    category_frame,
                    outcome,
                    "treat_cat",
                    category_x,
                    category_splits,
                    category_cluster_splits,
                ),
            )

            for index, row in (
                bundle["contrast"].summary.iterrows()
            ):
                level = int(
                    float(str(index).split(" vs ")[0])
                )

                loco_rows.append(
                    {
                        "sample": sample_name,
                        "outcome": outcome,
                        "comparison": (
                            f"{LEVELS[level]} "
                            "vs no treatment"
                        ),
                        "treatment_code": level,
                        "treatment_category": LEVELS[level],
                        "excluded_country": country,
                        "observations": len(category_frame),
                        "PSUs": (
                            category_frame[
                                "_psu_id"
                            ].nunique()
                        ),
                        "households": (
                            category_frame[
                                "_hh_id"
                            ].nunique()
                        ),
                        "coefficient": row.get(
                            "coef",
                            np.nan,
                        ),
                        "standard_error": row.get(
                            "std err",
                            np.nan,
                        ),
                        "p_value": row.get(
                            "P>|t|",
                            np.nan,
                        ),
                    }
                )

loco_results = pd.DataFrame(loco_rows)
loco_results.to_csv(
    OUT / "results_leave_one_country_out.csv",
    index=False,
)

display(loco_results.head())


### LOCO influence relative to the full estimate


In [ ]:
full_lookup = main_results.set_index(
    ["outcome", "comparison"]
)[["coefficient", "standard_error"]]

loco_influence = (
    loco_results.dropna(subset=["coefficient"])
    .join(
        full_lookup,
        on=["outcome", "comparison"],
        rsuffix="_full",
    )
)

loco_influence["change_from_full"] = (
    loco_influence["coefficient"]
    - loco_influence["coefficient_full"]
)

loco_influence["change_in_full_SE"] = (
    loco_influence["change_from_full"]
    / loco_influence["standard_error_full"]
)

loco_influence.to_csv(
    OUT / "results_leave_one_country_out_influence.csv",
    index=False,
)

display(
    loco_influence.sort_values(
        "change_in_full_SE",
        key=lambda values: values.abs(),
        ascending=False,
    ).head(20)
)


## Output manifest


In [ ]:
output_manifest = {
    "cleaning_diagnostics": str(
        OUT / "cleaning_diagnostics.csv"
    ),
    "treatment_frequencies": str(
        OUT / "treatment_category_frequencies.csv"
    ),
    "PSU_structure": str(
        OUT / "psu_structure_summary.csv"
    ),
    "support_by_country": str(
        OUT / "positivity_support_by_country.csv"
    ),
    "support_summary": str(
        OUT / "positivity_support_summary.csv"
    ),
    "fold_audit": str(
        OUT / "fold_balance_grouped_convex_sl.csv"
    ),
    "raw_propensities": str(
        OUT / "propensity_oof_raw_and_clipped.csv"
    ),
    "propensity_summary": str(
        OUT / "propensity_oof_summary.csv"
    ),
    "main_results": str(
        OUT / "results_main_grouped_convex_sl.csv"
    ),
    "support_restricted_results": str(
        OUT / "results_support_restricted.csv"
    ),
    "LOCO_results": str(
        OUT / "results_leave_one_country_out.csv"
    ),
}

manifest_path = OUT / "grouped_convex_sl_manifest.json"
manifest_path.write_text(
    json.dumps(output_manifest, indent=2),
    encoding="utf-8",
)

display(pd.Series(output_manifest, name="path"))
